In [1]:
import subprocess
print(subprocess.run(["rm", "-rf", "/home/sagemaker-user/.local/lib/python3.12/site-packages"], capture_output=True, text=True))
print("Cleaned.")

CompletedProcess(args=['rm', '-rf', '/home/sagemaker-user/.local/lib/python3.12/site-packages'], returncode=0, stdout='', stderr='')
Cleaned.


In [1]:
import numpy, pandas, boto3
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
import sagemaker
print("sagemaker attrs:", [a for a in dir(sagemaker) if not a.startswith('_')][:20])

numpy: 2.5.1
pandas: 3.0.5


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


sagemaker attrs: ['AlgorithmEstimator', 'AutoML', 'AutoMLDataChannel', 'AutoMLImageClassificationConfig', 'AutoMLInput', 'AutoMLJob', 'AutoMLJobV2', 'AutoMLTabularConfig', 'AutoMLTextClassificationConfig', 'AutoMLTextGenerationConfig', 'AutoMLTimeSeriesForecastingConfig', 'AutoMLV2', 'CandidateEstimator', 'CandidateStep', 'ContainerBaseModel', 'FactorizationMachines', 'FactorizationMachinesModel', 'FactorizationMachinesPredictor', 'FileSource', 'HyperparameterTuningJobAnalytics']


In [2]:
import boto3
import pandas as pd
import sagemaker

session = sagemaker.Session()
region = session.boto_region_name
bucket = "hassco-predictive-maintenance-ai-model2"
prefix = "processed/ai4i2020"

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
csv_key = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".csv")][0]
print(f"Found file: {csv_key}")

df = pd.read_csv(f"s3://{bucket}/{csv_key}")
print(df.shape)
df.head()

Found file: processed/ai4i2020/output_781d97ac-5002-4e45-bacd-117c72616400/part-00000-90e657d3-5f9d-48a9-9bcb-41c61726b9f9-c000.csv


(10000, 7)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [3]:
class_counts = df["Machine failure"].value_counts()
print(class_counts)

negative = class_counts[0]
positive = class_counts[1]
scale_pos_weight = negative / positive

print(f"\nNegative (no failure): {negative}")
print(f"Positive (failure): {positive}")
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

Machine failure
0    9661
1     339
Name: count, dtype: int64

Negative (no failure): 9661
Positive (failure): 339
scale_pos_weight = 28.50


In [4]:
from sklearn.model_selection import train_test_split

# Textspalte "Type" in Zahlen umwandeln (One-Hot-Encoding)
df_encoded = pd.get_dummies(df, columns=["Type"], drop_first=True)

# Zielspalte an den Anfang verschieben (Voraussetzung für XGBoost in SageMaker)
target = "Machine failure"
cols = [target] + [c for c in df_encoded.columns if c != target]
df_encoded = df_encoded[cols]

# Daten aufteilen: 70% Training, 15% Validierung, 15% Test (Klassenverhältnis bleibt erhalten)
train, temp = train_test_split(df_encoded, test_size=0.3, stratify=df_encoded[target], random_state=42)
validation, test = train_test_split(temp, test_size=0.5, stratify=temp[target], random_state=42)

print("Train:", train.shape, "| failures:", train[target].sum())
print("Validation:", validation.shape, "| failures:", validation[target].sum())
print("Test:", test.shape, "| failures:", test[target].sum())

train.head()

Train: (7000, 8) | failures: 237
Validation: (1500, 8) | failures: 51
Test: (1500, 8) | failures: 51


,Machine failure,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M
1888,0,297.8,307.4,1902,24.3,129,False,True
4858,0,303.7,312.3,1349,51.0,105,True,False
8990,0,297.2,307.9,1493,38.4,146,True,False
4901,0,303.6,312.3,1630,32.4,223,False,True
7957,0,300.9,311.9,2140,16.5,43,False,False


In [5]:
# Boolesche Werte in Zahlen umwandeln (XGBoost benötigt numerische Werte)
train = train.astype({c: int for c in train.columns if train[c].dtype == bool})
validation = validation.astype({c: int for c in validation.columns if validation[c].dtype == bool})
test = test.astype({c: int for c in test.columns if test[c].dtype == bool})

# Lokal als CSV speichern (ohne Header und ohne Index - Anforderung von SageMaker XGBoost)
train.to_csv("train.csv", header=False, index=False)
validation.to_csv("validation.csv", header=False, index=False)
test.to_csv("test.csv", header=False, index=False)

# Dateien nach S3 hochladen
xgb_prefix = "xgboost/data"
train_path = session.upload_data("train.csv", bucket=bucket, key_prefix=f"{xgb_prefix}/train")
validation_path = session.upload_data("validation.csv", bucket=bucket, key_prefix=f"{xgb_prefix}/validation")
test_path = session.upload_data("test.csv", bucket=bucket, key_prefix=f"{xgb_prefix}/test")

print("Train:", train_path)
print("Validation:", validation_path)
print("Test:", test_path)

Train: s3://hassco-predictive-maintenance-ai-model2/xgboost/data/train/train.csv
Validation: s3://hassco-predictive-maintenance-ai-model2/xgboost/data/validation/validation.csv
Test: s3://hassco-predictive-maintenance-ai-model2/xgboost/data/test/test.csv


In [7]:
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris

role = sagemaker.get_execution_role()

# Offizieller XGBoost-Container von AWS abrufen
container = image_uris.retrieve("xgboost", region, version="1.7-1")

# Estimator definieren (Trainingsjob-Konfiguration)
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=f"s3://{bucket}/xgboost/output",
    sagemaker_session=session,
)

# Hyperparameter setzen - scale_pos_weight behandelt das Klassenungleichgewicht
xgb.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="aucpr",          # besser als AUC bei unausgewogenen Daten
    scale_pos_weight=28.5,        # unser berechneter Wert
    num_round=200,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
)

# Datenkanäle definieren
train_input = TrainingInput(train_path, content_type="text/csv")
validation_input = TrainingInput(validation_path, content_type="text/csv")

# Training starten
xgb.fit({"train": train_input, "validation": validation_input})

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-07-26-09-09-51-545


ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:36                                                                                   │
│                                                                                                  │
│   33 validation_input = TrainingInput(validation_path, content_type="text/csv")                  │
│   34                                                                                             │
│   35 # Training starten                                                                          │
│ ❱ 36 xgb.fit({"train": train_input, "validation": validation_input})                             │
│   37                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator

In [8]:
import boto3

quotas = boto3.client("service-quotas", region_name=region)
paginator = quotas.get_paginator("list_service_quotas")

available = []
for page in paginator.paginate(ServiceCode="sagemaker"):
    for q in page["Quotas"]:
        if "training job usage" in q["QuotaName"] and q["Value"] > 0:
            available.append((q["QuotaName"], q["Value"]))

for name, val in sorted(available):
    print(f"{val:.0f}  -  {name}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│    4 paginator = quotas.get_paginator("list_service_quotas")                                     │
│    5                                                                                             │
│    6 available = []                                                                              │
│ ❱  7 for page in paginator.paginate(ServiceCode="sagemaker"):                                    │
│    8 │   for q in page["Quotas"]:                                                                │
│    9 │   │   if "training job usage" in q["QuotaName"] and q["Value"] > 0:                       │
│   10 │   │   │   available.append((q["QuotaName"], q["Value"]))                                  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/paginate.py:272 in __iter__                     │
│                                                                                                  │
│   269 │   │   starting_truncation = 0                                                            │
│   270 │   │   self._inject_starting_params(current_kwargs)                                       │
│   271 │   │   while True:                                                                        │
│ ❱ 272 │   │   │   response = self._make_request(current_kwargs)                                  │
│   273 │   │   │   parsed = self._extract_parsed_response(response)                               │
│   274 │   │   │   if first_request:                                                              │
│   275 │   │   │   │   # The first request is handled differently.  We could                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/paginate.py:360 in _make_request                │
│                                                                                                  │
│   357 │                                                                                          │
│   358 │   @with_current_context(partial(register_feature_id, 'PAGINATOR'))                       │
│   359 │   def _make_request(self, current_kwargs):                                               │
│ ❱ 360 │   │   return self._method(**current_kwargs)                                              │
│   361 │                                                                                          │
│   362 │   def _extract_parsed_response(self, response):                                          │
│   363 │   │   return response                              

In [9]:
import xgboost as xgb_lib
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report

# Features und Zielspalte trennen
X_train, y_train = train.iloc[:, 1:], train.iloc[:, 0]
X_val, y_val = validation.iloc[:, 1:], validation.iloc[:, 0]
X_test, y_test = test.iloc[:, 1:], test.iloc[:, 0]

# Modell mit scale_pos_weight trainieren
model = xgb_lib.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=28.5,        # unser berechneter Wert gegen Klassenungleichgewicht
    n_estimators=200,
    max_depth=5,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Auf Testdaten auswerten
y_pred = model.predict(X_test)

print("=== XGBoost mit scale_pos_weight=28.5 ===")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["Kein Ausfall", "Ausfall"]))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:22                                                                                   │
│                                                                                                  │
│   19 │   random_state=42,                                                                        │
│   20 )                                                                                           │
│   21                                                                                             │
│ ❱ 22 model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)                       │
│   23                                                                                             │
│   24 # Auf Testdaten auswerten                                                                   │
│   25 y_pred = model.predict(X_test)                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/xgboost/core.py:726 in inner_f                           │
│                                                                                                  │
│    723 │   │   │   │   warnings.warn(msg, FutureWarning)                                         │
│    724 │   │   │   for k, arg in zip(sig.parameters, args):                                      │
│    725 │   │   │   │   kwargs[k] = arg                                                           │
│ ❱  726 │   │   │   return func(**kwargs)                                                         │
│    727 │   │                                                                                     │
│    728 │   │   return inner_f                                                                    │
│    729                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/xgboost/sklearn.py:1580 in fit                           │
│                                                                                                  │
│   1577 │   │   │   │   params["num_class"] = self.n_classes_                                     │
│   1578 │   │   │                                                                                 │
│   1579 │   │   │   model, metric, params = self._configure_fit(xgb_model, params)                │
│ ❱ 1580 │   │   │   train_dmatrix, evals = _wrap_evaluation_matrices(                             │
│   1581 │   │   │   │   missing=self.missing,                                                     │
│   1582 │   │   │   │   X=X,                                                                      │
│   1583 │   │   │   │   y=y,                                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/xgboost/sklearn.py:603 in _wrap_evaluation_matrices      │
│                                                                                                  │
│    600 ) -> Tuple[Any, List[Tuple[Any, str]]]:                                                   │
│    601 │   """Convert array_like evaluation matrices into DMatrix.  Perform validation on the    │
│    602 │   way."""                                                                               │
│ ❱  603 │   train_dmatrix = create_dmatrix(                                                       │
│    604 │   │   data=X,                                                                           │
│    605 │   │   label=y,                                                                          │
│    606 │   │   group=group,                                

In [10]:
import re

# Spaltennamen bereinigen (XGBoost erlaubt keine [, ] oder < in Namen)
def clean_names(dfx):
    dfx = dfx.copy()
    dfx.columns = [re.sub(r"[\[\]<>]", "", str(c)).strip().replace(" ", "_") for c in dfx.columns]
    return dfx

train_c = clean_names(train)
validation_c = clean_names(validation)
test_c = clean_names(test)

print(list(train_c.columns))

['Machine_failure', 'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Type_L', 'Type_M']


In [11]:
import xgboost as xgb_lib
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report

X_train, y_train = train_c.iloc[:, 1:], train_c.iloc[:, 0]
X_val, y_val = validation_c.iloc[:, 1:], validation_c.iloc[:, 0]
X_test, y_test = test_c.iloc[:, 1:], test_c.iloc[:, 0]

model = xgb_lib.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=28.5,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred = model.predict(X_test)

print("=== XGBoost mit scale_pos_weight=28.5 ===")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["Kein Ausfall", "Ausfall"]))

=== XGBoost mit scale_pos_weight=28.5 ===
F1:        0.716
Precision: 0.672
Recall:    0.765

Confusion Matrix:
[[1430   19]
 [  12   39]]

               precision    recall  f1-score   support

Kein Ausfall       0.99      0.99      0.99      1449
     Ausfall       0.67      0.76      0.72        51

    accuracy                           0.98      1500
   macro avg       0.83      0.88      0.85      1500
weighted avg       0.98      0.98      0.98      1500



In [12]:
# Verschiedene scale_pos_weight-Werte testen
for w in [1, 3, 5, 10, 15, 20, 28.5]:
    m = xgb_lib.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr",
        scale_pos_weight=w, n_estimators=200, max_depth=5,
        learning_rate=0.2, subsample=0.8, colsample_bytree=0.8, random_state=42,
    )
    m.fit(X_train, y_train, verbose=False)
    p = m.predict(X_test)
    print(f"weight={w:>5}  F1={f1_score(y_test,p):.3f}  Precision={precision_score(y_test,p):.3f}  Recall={recall_score(y_test,p):.3f}")

weight=    1  F1=0.729  Precision=0.912  Recall=0.608


weight=    3  F1=0.745  Precision=0.814  Recall=0.686


weight=    5  F1=0.727  Precision=0.750  Recall=0.706


weight=   10  F1=0.720  Precision=0.735  Recall=0.706


weight=   15  F1=0.725  Precision=0.725  Recall=0.725


weight=   20  F1=0.745  Precision=0.745  Recall=0.745


weight= 28.5  F1=0.716  Precision=0.672  Recall=0.765


In [13]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score

# Suchraum definieren
param_dist = {
    "scale_pos_weight": [3, 5, 10, 15, 20, 25],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "n_estimators": [100, 200, 300, 500],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.5],
}

base_model = xgb_lib.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
)

# Zufällige Suche mit Kreuzvalidierung (optimiert auf F1 der Ausfallklasse)
search = RandomizedSearchCV(
    base_model,
    param_distributions=param_dist,
    n_iter=100,                  # 100 zufällige Kombinationen testen
    scoring=make_scorer(f1_score),
    cv=3,                        # 3-fache Kreuzvalidierung
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

search.fit(X_train, y_train)

print("Beste Parameter:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBester F1 (Kreuzvalidierung): {search.best_score_:.3f}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits


Beste Parameter:
  subsample: 1.0
  scale_pos_weight: 25
  n_estimators: 100
  min_child_weight: 1
  max_depth: 8
  learning_rate: 0.3
  gamma: 0.1
  colsample_bytree: 1.0

Bester F1 (Kreuzvalidierung): 0.735


In [14]:
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("=== Optimiertes XGBoost ===")
print(f"F1:        {f1_score(y_test, y_pred_best):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_best):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))
print("\n=== Vergleich ===")
print(f"Autopilot:              F1=0.820  Precision=0.926  Recall=0.735")
print(f"XGBoost (weight=28.5):  F1=0.716  Precision=0.672  Recall=0.765")
print(f"XGBoost (optimiert):    F1={f1_score(y_test,y_pred_best):.3f}  Precision={precision_score(y_test,y_pred_best):.3f}  Recall={recall_score(y_test,y_pred_best):.3f}")

=== Optimiertes XGBoost ===
F1:        0.718
Precision: 0.712
Recall:    0.725

Confusion Matrix:
[[1434   15]
 [  14   37]]

=== Vergleich ===
Autopilot:              F1=0.820  Precision=0.926  Recall=0.735
XGBoost (weight=28.5):  F1=0.716  Precision=0.672  Recall=0.765
XGBoost (optimiert):    F1=0.718  Precision=0.712  Recall=0.725


In [15]:
def add_physics_features(dfx):
    """Physikbasierte Features basierend auf den bekannten Ausfallmechanismen des AI4I-Datensatzes"""
    d = dfx.copy()
    
    # Leistung = Drehmoment x Winkelgeschwindigkeit (relevant für PWF - Power Failure)
    d["Power_W"] = d["Torque_Nm"] * d["Rotational_speed_rpm"] * 2 * 3.14159 / 60
    
    # Temperaturdifferenz (relevant für HDF - Heat Dissipation Failure)
    d["Temp_diff_K"] = d["Process_temperature_K"] - d["Air_temperature_K"]
    
    # Verschleiß x Drehmoment (relevant für OSF - Overstrain Failure)
    d["Wear_x_Torque"] = d["Tool_wear_min"] * d["Torque_Nm"]
    
    # Zusätzliche Interaktionen
    d["Torque_per_rpm"] = d["Torque_Nm"] / d["Rotational_speed_rpm"]
    d["Wear_squared"] = d["Tool_wear_min"] ** 2
    
    return d

train_f = add_physics_features(train_c)
val_f = add_physics_features(validation_c)
test_f = add_physics_features(test_c)

print("Neue Spalten:", [c for c in train_f.columns if c not in train_c.columns])
print("Gesamt-Features:", train_f.shape[1] - 1)

Neue Spalten: ['Power_W', 'Temp_diff_K', 'Wear_x_Torque', 'Torque_per_rpm', 'Wear_squared']
Gesamt-Features: 12


In [16]:
X_train_f, y_train_f = train_f.drop(columns=["Machine_failure"]), train_f["Machine_failure"]
X_val_f, y_val_f = val_f.drop(columns=["Machine_failure"]), val_f["Machine_failure"]
X_test_f, y_test_f = test_f.drop(columns=["Machine_failure"]), test_f["Machine_failure"]

# Erneute Hyperparameter-Suche mit den neuen Features
search_f = RandomizedSearchCV(
    xgb_lib.XGBClassifier(objective="binary:logistic", eval_metric="aucpr", random_state=42),
    param_distributions=param_dist,
    n_iter=100,
    scoring=make_scorer(f1_score),
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)
search_f.fit(X_train_f, y_train_f)

best_f = search_f.best_estimator_
y_pred_f = best_f.predict(X_test_f)

print("\n=== XGBoost + Physik-Features ===")
print(f"F1:        {f1_score(y_test_f, y_pred_f):.3f}")
print(f"Precision: {precision_score(y_test_f, y_pred_f):.3f}")
print(f"Recall:    {recall_score(y_test_f, y_pred_f):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_f, y_pred_f))

print("\n=== Gesamtvergleich ===")
print(f"Autopilot:                    F1=0.820  Precision=0.926  Recall=0.735")
print(f"XGBoost (Rohdaten, optimiert): F1=0.718  Precision=0.712  Recall=0.725")
print(f"XGBoost + Physik-Features:     F1={f1_score(y_test_f,y_pred_f):.3f}  Precision={precision_score(y_test_f,y_pred_f):.3f}  Recall={recall_score(y_test_f,y_pred_f):.3f}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits



=== XGBoost + Physik-Features ===
F1:        0.860
Precision: 0.878
Recall:    0.843

Confusion Matrix:
[[1443    6]
 [   8   43]]

=== Gesamtvergleich ===
Autopilot:                    F1=0.820  Precision=0.926  Recall=0.735
XGBoost (Rohdaten, optimiert): F1=0.718  Precision=0.712  Recall=0.725
XGBoost + Physik-Features:     F1=0.860  Precision=0.878  Recall=0.843
